In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QUICK-PDE: Qiskit Function od ColibriTD
> **Note:** Qiskit Functions jsou experimentální funkce dostupné uživatelům IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API) Plan. Jsou ve stavu náhledového vydání a mohou se změnit.
## Přehled
Řešič parciálních diferenciálních rovnic (PDE) představený zde je součástí naší platformy Quantum Innovative Computing Kit (QUICK) (QUICK-PDE) a je zabalen jako Qiskit Function. S funkcí QUICK-PDE můžeš řešit doménově specifické parciální diferenciální rovnice na QPU IBM Quantum. Tato funkce je založena na algoritmu popsaném v [dokumentu H-DES od ColibriTD.](https://arxiv.org/abs/2410.01130) Tento algoritmus dokáže řešit složité multifyzikální problémy, počínaje výpočetní dynamikou tekutin (CFD) a deformací materiálů (MD), přičemž další případy použití přibývají.

Při řešení diferenciálních rovnic jsou zkušební řešení zakódována jako lineární kombinace ortogonálních funkcí (typicky Čebyševovy polynomy, konkrétně $2^n$ z nich, kde $n$ je počet qubitů kódujících tvou funkci), parametrizované úhly Variable Quantum Circuit (VQC). Ansatz generuje stav kódující funkci, který je vyhodnocován pozorovatelnými, jejichž kombinace umožňují vyhodnotit funkci ve všech bodech. Poté můžeš vyhodnotit ztrátovou funkci, do které jsou zakódovány diferenciální rovnice, a doladit úhly v hybridní smyčce, jak je znázorněno níže. Zkušební řešení se postupně přibližují skutečným řešením, dokud nedosáhneš uspokojivého výsledku.

![Pracovní postup funkce QUICK-PDE](../docs/images/guides/colibritd-equation-solver/diagram.svg)

Kromě této hybridní smyčky můžeš také řetězit různé optimalizátory dohromady. To je užitečné, když chceš, aby globální optimalizátor nalezl dobrou sadu úhlů, a poté jemnější optimalizátor sledoval gradient k nejlepší sadě sousedních úhlů. V případě výpočetní dynamiky tekutin (CFD) výchozí optimalizační sekvence přináší nejlepší výsledky – v případě deformace materiálů (MD) sice výchozí nastavení poskytuje dobré výsledky, ale můžeš ho dále konfigurovat pro výhody specifické pro daný problém.

Poznámka: pro každou proměnnou funkce specifikujeme počet qubitů (se kterým si můžeš hrát). Naskládáním 10 identických Circuit a vyhodnocením 10 identických pozorovatelných na různých qubitech v rámci jednoho velkého Circuit můžeš provádět potlačení šumu v rámci procesu optimalizace CMA, spoléhajíce na metodu noise learner, a výrazně snížit počet potřebných měření.
## Vstup/výstup
### Výpočetní dynamika tekutin
Neviskózní Burgersova rovnice modeluje proudění neviskózních tekutin takto:

$$\frac{\partial u}{\partial t} + u\frac{\partial u}{\partial x} = 0,$$

$u$ představuje pole rychlosti tekutiny. Tento případ použití má časovou okrajovou podmínku: můžeš vybrat počáteční podmínku a pak nechat systém relaxovat. V současnosti jsou přijímány pouze lineární funkce jako počáteční podmínky: $ax + b$.

Argumenty diferenciálních rovnic pro CFD jsou na pevné mřížce, jak následuje:

- $t$ je v rozsahu 0 až 0,95 s 30 vzorkovacími body. $x$ je v rozsahu 0 až 0,95 s krokem 0,2375.

### Deformace materiálů
Tento případ použití se zaměřuje na hypoelastickou deformaci s jednorozměrným zkouškou tahem, při níž je tyč pevně uchycena v prostoru a táhnuta na druhém konci. Popis problému je následující:

$$u' - \frac{\sigma}{3K} - \frac{2}{\sqrt{3}}\epsilon_0\left(\frac{\sigma'}{\sigma_0\sqrt{3}}\right)^n = 0$$

$$\sigma' - b = 0,$$

$K$ představuje objemový modul pružnosti roztahovaného materiálu, $n$ exponent mocninného zákona, $b$ sílu na jednotku hmotnosti, $\epsilon_0$ proporcionální mez napětí, $\sigma_0$ proporcionální mez přetvoření, $u$ funkci napětí a $\sigma$ funkci přetvoření.

Uvažovaná tyč má jednotkovou délku. Tento případ použití má okrajovou podmínku pro povrchové napětí $t$, tedy množství práce potřebné k natažení tyče.

Argumenty diferenciálních rovnic pro MD jsou na pevné mřížce, jak následuje:

- $x$ je v rozsahu 0 až 1 s krokem 0,04.
### Vstup
Pro spuštění Qiskit Function QUICK-PDE můžeš upravit následující parametry:

| Název             | Typ                                                  | Popis                                                                                                                                                                                                                                                                           | Specifické pro případ použití | Příklad                  |
| ----------------- | ---------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------- | ------------------------ |
| use_case          | `Literal["MD", "CFD"]`                               | Přepínač pro výběr soustavy diferenciálních rovnic k řešení                                                                                                                                                                                                                     | Ne                            | `"CFD"`                  |
| parameters        | `dict[str, Any]`                                     | Parametry diferenciální rovnice (viz následující tabulka pro více podrobností)                                                                                                                                                                                                  | Ne                            | `{"a": 1.0, "b": 1.0}`   |
| nb_qubits         | `Optional[dict[str, dict[str, int]]]`                | Počet qubitů na funkci a proměnnou. Funkce volí optimalizované hodnoty, ale pokud chceš zkusit najít lepší kombinaci, můžeš přepsat výchozí hodnoty                                                                                                                              | Ne                            | `{"u": {"x": 1, "t":3}}` |
| depth             | `Optional[dict[str, int]]`                           | Hloubka ansatzu na funkci. Funkce volí optimalizované hodnoty, ale pokud chceš zkusit najít lepší kombinaci, můžeš přepsat výchozí hodnoty                                                                                                                                       | Ne                            | `{"u": 4}`               |
| optimizer         | `Optional[list[str]]`                                | Optimalizátory, které se mají použít: buď "CMAES" z [`cma` python knihovny](https://github.com/CMA-ES/pycma) nebo jeden z [optimalizátorů scipy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html)                                             | MD                            | `"SLSQP"`                |
| shots             | `Optional[list[int]]`                                | Počet měření použitých ke spuštění každého Circuit. Protože je potřeba několik kroků optimalizace, délka seznamu musí být rovna počtu použitých optimalizátorů (4 v případě CFD). Výchozí hodnota je `[50_000] * nb_optimizers` pro MD a `[5_00, 2_000, 5_000, 10_000]` pro CFD | Ne                            | `[15_000, 30_000]`       |
| optimizer_options | `Optional[dict[str, Any]]`                           | Možnosti předávané optimalizátoru. Podrobnosti tohoto vstupu závisí na použitém optimalizátoru; pro konkrétní informace viz dokumentaci příslušného optimalizátoru                                                                                                               | MD                            | `{"maxiter": 50 }`       |
| initialization    | `Optional[Literal["RANDOM", "PHYSICALLY_INFORMED"]]` | Zda začít s náhodnými úhly nebo chytře zvolenými úhly. Upozornění: chytře zvolené úhly nemusí fungovat ve 100 % případů. Výchozí hodnota je `"PHYSICALLY_INFORMED"`.                                                                                                             | Ne                            | `"RANDOM"`               |
| backend_name      | `Optional[str]`                                      | Název Backend, který se má použít.                                                                                                                                                                                                                                              | Ne                            | `"ibm_torino"`           |
| mode              | `Optional[Literal["job", "session", "batch"]]`        | Režim provádění, který se má použít. Výchozí hodnota je `"job"`.                                                                                                                                                                                                                | Ne                            | `"job"`                  |

Parametry diferenciální rovnice (fyzikální parametry a okrajová podmínka) by měly mít následující formát:

| Případ použití | Klíč        | Typ hodnoty | Popis                                        | Příklad |
| -------------- | ----------- | ----------- | -------------------------------------------- | ------- |
| CFD            | `a`         | `float`     | Koeficient počátečních hodnot $u$            | `1.0`   |
| CFD            | `b`         | `float`     | Offset počátečních hodnot $u$                | `1.0`   |
| MD             | `t`         | `float`     | povrchové napětí                             | `12.0`  |
| MD             | `K`         | `float`     | objemový modul pružnosti                     | `100.0` |
| MD             | `n`         | `int`       | mocninný zákon                               | `4.0`   |
| MD             | `b`         | `float`     | síla na jednotku hmotnosti                   | `10.0`  |
| MD             | `epsilon_0` | `float`     | proporcionální mez napětí                    | `0.1`   |
| MD             | `sigma_0`   | `float`     | proporcionální mez přetvoření                | `5.0`   |
### Výstup
Výstupem je slovník se seznamem vzorkovacích bodů a hodnotami funkcí v každém z těchto bodů:

In [ ]:
from numpy import array

In [ ]:
solution = {
    "functions": {
        "u": array(
            [
                [0.01, 0.1, 1],
                [0.02, 0.2, 2],
                [0.03, 0.3, 3],
                [0.04, 0.4, 4],
            ]
        ),
    },
    "samples": {
        "t": array([0.1, 0.2, 0.3, 0.4]),
        "x": array([0.5, 0.6, 0.7]),
    },
}

Tvar pole řešení závisí na vzorcích proměnných:

In [ ]:
assert len(solution["functions"]["u"].shape) == len(solution["samples"])
for col_size, samples in zip(
    solution["functions"]["u"].shape, solution["samples"].values()
):
    assert col_size == len(samples)

Mapování mezi vzorkovacími body proměnných funkce a dimenzí pole řešení se provádí v alfanumerickém pořadí názvů proměnných. Například pokud jsou proměnné `"t"` a `"x"`, řádek `solution["functions"]["u"]` představuje hodnoty řešení pro pevné `"t"` a sloupec `solution["functions"]["u"]` představuje hodnoty řešení pro pevné `"x"`.

Následuje příklad, jak získat hodnotu funkce pro konkrétní sadu souřadnic:

In [ ]:
# u(t=0.2, x=0.7) == 2
assert solution["samples"]["t"][1] == 0.2
assert solution["samples"]["x"][2] == 0.7
assert solution["functions"]["u"][1, 2] == 2

## Benchmarky

Následující tabulka uvádí statistiky různých spuštění naší funkce.

| Příklad                           | Počet qubitů | Inicializace          | Chyba     | Celkový čas (min) | Využití runtime (min) |
| --------------------------------- | ------------ | --------------------- | --------- | ----------------- | --------------------- |
| Neviskózní Burgersova rovnice     | 50           | `PHYSICALLY_INFORMED` | $10^{-2}$ | 66                | 25                    |
| Hypoelastická 1D zkouška tahem    | 18           | `RANDOM`              | $10^{-2}$ | 123               | 100                   |

## Začínáme

Vyplň [formulář pro vyžádání přístupu k funkci QUICK-PDE.](https://forms.cloud.microsoft/e/3Wi9cbjQPK) Poté, za předpokladu, že sis již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do lokálního prostředí, vyber funkci takto:

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

quick = catalog.load("colibritd/quick-pde")

## Příklady

Pro začátek vyzkoušej jeden z následujících příkladů:

### Výpočetní dynamika tekutin (CFD)

Když jsou počáteční podmínky nastaveny na $u(0,x) = x$, výsledky jsou následující:

In [ ]:
# launch the simulation with initial conditions u(0,x) = a*x + b
job = quick.run(use_case="cfd", physical_parameters={"a": 1.0, "b": 0.0})

Zkontroluj [stav](/guides/functions#check-job-status) úlohy své Qiskit Function nebo získej [výsledky](/guides/functions#retrieve-results) takto:

In [ ]:
print(job.status())
solution = job.result()

'QUEUED'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

_ = plt.figure()
ax = plt.axes(projection="3d")

# plot the solution using the 3d plotting capabilities of pyplot
t, x = np.meshgrid(solution["samples"]["t"], solution["samples"]["x"])
ax.plot_surface(
    t,
    x,
    solution["functions"]["u"],
    edgecolor="royalblue",
    lw=0.25,
    rstride=26,
    cstride=26,
    alpha=0.3,
)
ax.scatter(t, x, solution["functions"]["u"], marker=".")
ax.set(xlabel="t", ylabel="x", zlabel="u(t,x)")

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif" alt="Output of the previous code cell" />

### Material Deformation

The material deformation use case requires the physical parameters of your material and the applied force, as follows:

In [ ]:
import matplotlib.pyplot as plt

# select the properties of your material
job = quick.run(
    use_case="md",
    physical_parameters={
        "t": 12.0,
        "K": 100.0,
        "n": 4.0,
        "b": 10.0,
        "epsilon_0": 0.1,
        "sigma_0": 5.0,
    },
)

# plot the result
solution = job.result()

_ = plt.figure()
stress_plot = plt.subplot(211)
plt.plot(solution["samples"]["x"], solution["functions"]["u"])
strain_plot = plt.subplot(212)
plt.plot(solution["samples"]["x"], solution["functions"]["sigma"])

plt.show()

<Image src="../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/colibritd-pde/extracted-outputs/c42aba9b-0.avif)

### Deformace materiálů
Případ použití deformace materiálů vyžaduje fyzikální parametry tvého materiálu a přiloženou sílu, jak následuje:

In [ ]:
job = quick.run(use_case="mdf", physical_params={})

print(job.error_message())


# or write a wrapper around it for a more human readable version
def pprint_error(job):
    print("".join(eval(job.error_message())["error"]))


print("___")
pprint_error(job)

{"error": ["qiskit.exceptions.QiskitError: 'Unknown argument \"physical_params\", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'\n"]}
___
qiskit.exceptions.QiskitError: 'Unknown argument "physical_params", did you make a typo? -- https://docs.quantum.ibm.com/errors#1804'



![Výstup předchozí buňky kódu](../docs/images/guides/colibritd-pde/extracted-outputs/a568e325-0.avif)

## Načítání chybových zpráv

Pokud je stav tvé úlohy `ERROR`, použij `job.error_message()` k načtení chybové zprávy pro pomoc při ladění, jak následuje: